In [100]:
import sys
sys.path.append(r'C:\Users\liuyujie714\Desktop\TprParser\x64\Release')

import subprocess, shutil, copy
import TprParser
import numpy as np

class TprReader:
    """ @brief A wrapper of TprParser
        1. get atmic coordinates of tpr
        2. modify simulation nsteps, delta t or coordinates and save as new.tpr
    """
    def __init__(self, fname, bGRO = False, bMol2 = False, bCharge = False) -> None:
        # get internal object
        self.tprCapsule = TprParser.load(fname, bGRO, bMol2, bCharge)
    
    def set_nsteps(self, nsteps):
        """ @brief set up nsteps of tpr, same as mdp

        Parameters
        ----------
        nsteps: the nsteps of simulation

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_nsteps(self.tprCapsule, nsteps)

    def set_dt(self, dt):
        """ @brief set up dt of tpr in ps, same as mdp

        Parameters
        ----------
        dt: the dt of simulation, ps

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_dt(self.tprCapsule, dt)
    
    def set_coords(self, coords:np.array):
        """ @brief set up atomic coordinates of tpr

        Parameters
        ----------
        coords: a np.array(dtype=np.float32) of atom coordinates, the dimension must be natoms * 3

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_coordinates(self.tprCapsule, np.array(coords, dtype=np.float32).flatten())
    
    def set_pressure(self, epc, epct, tau_p, ref_p, compress):
        """ @brief set up pressure coulping parts of tpr

        Parameters
        ----------
        epc: pressure coupling method, No, Berendsen, ParrinelloRahman, CRescale
        epct: pressure coupling type, Isotropic, SemiIsotropic
        tau_p: the pressure coupling constant
        ref_p: a list of pressure in bar, the length must be 9
        compress: a list of compressibility in bar^-1, the length must be 9

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_pressure(self.tprCapsule, epc, epct, tau_p, ref_p, compress)
    
    def set_temperature(self, etc, tau_t:list, ref_t:list):
        """ @brief set up temperature coulping parts of tpr

        Parameters
        ----------
        etc: temperature coupling method, No, Berendsen, NoseHoover, VRescale
        tau_t: the temperature coupling constant, the length must be same as old tpr
        ref_t: a list of temperature in bar, the length must be same as old tpr

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_temperature(self.tprCapsule, etc, tau_t, ref_t)

    def set_mdp_integer(self, keyword:str, val:int):
        """ @brief set up integer keyword of tpr

        Parameters
        ----------
        keyword: the mdp keyword, nstlog, nstxout, nstvout, nstfout, nstenergy, nstxout_compressed, \
            nsttcouple, nstpcouple, nstcalcenergy
        val: a int value for keyword

        Returns
        -------
        return True if succeed
        """
        return TprParser.set_mdp_integer(self.tprCapsule, keyword, val)
        
    def get_xvf(self, type:str) -> np.array:
        """ @brief get atomic coordinates/velocity/force from tpr if exist. \
        the unit is nm, nm/ps, kJ/mol/nm

        Parameters
        ----------
        type: must be 'X', 'V', or 'F', represents atomic coordinates/velocity/force to get

        Returns
        -------
        return a np.array(dtype=np.float32), the dimension is natoms * 3
        """
        vec = TprParser.get_xvf(self.tprCapsule, type)
        return np.array(vec).reshape(-1, 3)


In [101]:
def Pressure(fname):
    reader = TprReader(fname)
    ref_p = [
        100, 0, 0,
        0, 100, 0,
        0, 0, 100
    ]
    compress = [
        4.5E-5, 0, 0,
        0, 4.5E-5, 0,
        0, 0, 4.5E-5
    ]
    assert len(ref_p) == 9
    assert len(compress) == 9
    reader.set_pressure('No', 'Isotropic', 1.0, ref_p, compress)


In [102]:
def run_cmd(cmd:str):
    ret = subprocess.run(cmd, shell=True)
    if ret.returncode != 0:
        raise Exception('\nError occurred from command: \n\t%s!!!' %cmd)
    
def MD(inittpr:str, nsteps:int = 10):
    reader = TprReader(inittpr)
    coords = reader.get_xvf('X') # get coords from tpr
    natmA = 120
    natmB = 132
    natm = natmA+natmB

    assert natm == coords.shape[0]
    for i in range(nsteps):
        # move two molecules distance of Z axis each 2.0 A
        tempcoords = copy.deepcopy(coords)
        tempcoords[:natmA,     2] += 0.05 * i
        tempcoords[natmA:natm, 2] -= 0.05 * i
        reader.set_coords(tempcoords)
        # rename new.tpr to em_{i}.tpr
        suffix = inittpr.split(".tpr")[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx mdrun -deffnm {suffix} -v')
    print("Finished!")

In [103]:
def Temperature(fname:str, nsteps:int=2):
    tau_t = [1.0, 1.0] # coupling constant
    reader = TprReader(fname)
    for i in range(nsteps):
        ref_t = [100+i*100, 100+i*100] # 100, 200, 300 K
        reader.set_temperature("Vrescale", tau_t, ref_t)
        suffix = fname.split('.tpr')[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx5 mdrun -deffnm {suffix} -v')


In [104]:
def MDP_Integer(fname, key, val):
    reader = TprReader(fname)
    reader.set_mdp_integer(key, val)

In [105]:
def get_xvf(fname, type):
    reader = TprReader(fname)
    return reader.get_xvf(type)

In [106]:
if __name__ == '__main__':
    MD('test/em.tpr', 10)

Finished!
